# FluidSynth MIDI Timbre Pilot

This notebook takes the explicit MIDI-control idea from the previous pilot and renders it with a sample-based General MIDI SoundFont.

## What we compare
- One fixed MIDI arrangement rendered with three lead instruments: acoustic piano, marimba, and harp.
- The bass MIDI track remains the same in each version.
- The note pitches, scale, tempo, meter, beat positions, and note lengths stay fixed. Only the MIDI program/timbre changes.
- The piano version is also rendered as separate Lead and Bass stems.

## Scope and licensing
This notebook uses FluidSynth plus FluidR3 GM from the Debian package `fluid-soundfont-gm`. FluidSynth is LGPL-2.1; FluidR3's published upstream license is MIT. The package's copyright notice is included in the downloaded result bundle for traceability. This is a prototype renderer comparison, not a final review of sample-by-sample provenance for product distribution.

The SoundFont is installed in the Colab runtime. It is not committed to the GitHub repository or included in the results ZIP.

## Step 1 — Install the renderer and SoundFont

This is a one-time runtime setup. The SoundFont package is large (roughly 145 MB installed). No GPU, model checkpoint, dataset, or API key is needed.

In [ ]:
import subprocess
from pathlib import Path

def run_command(command, label):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-3000:])
        print(result.stderr[-5000:])
        raise RuntimeError(f"{label} failed with exit code {result.returncode}")
    return result

print("Updating package index...")
run_command(["apt-get", "update", "-qq"], "apt-get update")

print("Installing FluidSynth and FluidR3 GM...")
install = run_command(
    ["apt-get", "install", "-y", "-qq", "fluidsynth", "fluid-soundfont-gm"],
    "apt-get install",
)
print((install.stdout or "Install completed.")[-1500:])

SOUNDFONT = Path("/usr/share/sounds/sf2/FluidR3_GM.sf2")
if not SOUNDFONT.exists():
    candidates = list(Path("/usr/share").rglob("FluidR3_GM.sf2"))
    if not candidates:
        raise FileNotFoundError("FluidR3_GM.sf2 was not found after package installation.")
    SOUNDFONT = candidates[0]

print("FluidSynth:", subprocess.check_output(["fluidsynth", "--version"], text=True).splitlines()[0])
print("SoundFont:", SOUNDFONT)
print(f"SoundFont size: {SOUNDFONT.stat().st_size / (1024**2):.1f} MiB")

## Step 2 — Set the musical controls

The MIDI arrangement is explicit. The default is a four-bar C-major motif at 96 BPM in 4/4. You can change the tempo, key, or scale and rerun all later cells.

This notebook's main comparison keeps the musical controls fixed so that listening differences come from the instrument patch.

In [ ]:
import hashlib
import json
import math
import struct
import zipfile
import numpy as np
from scipy.io.wavfile import read as read_wav
from IPython.display import Audio, display

OUT_DIR = Path("/content/fluidsynth_midi_timbre_pilot")
OUT_DIR.mkdir(parents=True, exist_ok=True)

BPM = 96
KEY_NAME = "C"
SCALE_NAME = "major"
TONIC_MIDI = 60  # C4
SCALE_INTERVALS = [0, 2, 4, 5, 7, 9, 11]
METER = (4, 4)
BARS = 4
TICKS_PER_BEAT = 480
SAMPLE_RATE = 44100

# GM program numbers are zero-based.
LEAD_PRESETS = {
    "acoustic_piano": 0,
    "marimba": 12,
    "harp": 46,
}
BASS_PROGRAM = 32  # Acoustic Bass

print(f"Arrangement: {KEY_NAME} {SCALE_NAME}, {BPM} BPM, {METER[0]}/{METER[1]}, {BARS} bars")
print("Lead presets:", LEAD_PRESETS)
print("Bass program:", BASS_PROGRAM)

## Step 3 — Build the same note events

The event list is the source of truth. Every lead and bass note has a scale degree, pitch, beat position, duration, and velocity. Changing the instrument patch does not change any of these values.

In [ ]:
MELODY_PATTERN = [
    [(0.0, 0, 0.42, 92), (0.5, 2, 0.42, 76), (1.0, 4, 0.42, 84), (1.5, 2, 0.42, 72),
     (2.0, 3, 0.42, 86), (2.5, 4, 0.42, 78), (3.0, 2, 0.42, 74), (3.5, 1, 0.42, 70)],
    [(0.0, 4, 0.42, 88), (0.5, 5, 0.42, 80), (1.0, 6, 0.42, 84), (1.5, 5, 0.42, 74),
     (2.0, 4, 0.42, 86), (2.5, 2, 0.42, 76), (3.0, 3, 0.42, 78), (3.5, 2, 0.42, 70)],
    [(0.0, 0, 0.42, 90), (0.5, 2, 0.42, 76), (1.0, 4, 0.42, 84), (1.5, 6, 0.42, 78),
     (2.0, 5, 0.42, 86), (2.5, 3, 0.42, 74), (3.0, 4, 0.42, 80), (3.5, 2, 0.42, 72)],
    [(0.0, 1, 0.42, 84), (0.5, 3, 0.42, 76), (1.0, 5, 0.42, 82), (1.5, 4, 0.42, 74),
     (2.0, 2, 0.42, 84), (2.5, 1, 0.42, 72), (3.0, 0, 0.42, 80), (3.5, 0, 0.80, 92)],
]
BASS_ROOT_DEGREES = [0, 3, 4, 2]

def degree_offset(degree, intervals):
    size = len(intervals)
    if degree >= 0:
        octave, index = divmod(degree, size)
        return 12 * octave + intervals[index]
    return -degree_offset(-degree, intervals)

events = []
for bar in range(BARS):
    for beat_in_bar, degree, duration_beats, velocity in MELODY_PATTERN[bar]:
        events.append({
            "part": "Lead",
            "bar": bar,
            "beat_in_bar": beat_in_bar,
            "start_beat": bar * METER[0] + beat_in_bar,
            "degree": degree,
            "pitch": TONIC_MIDI + degree_offset(degree, SCALE_INTERVALS),
            "duration_beats": duration_beats,
            "velocity": velocity,
        })
    root = BASS_ROOT_DEGREES[bar]
    for beat_in_bar, degree, duration_beats, velocity in [
        (0.0, root, 1.75, 88),
        (2.0, root + 4, 1.75, 76),
    ]:
        events.append({
            "part": "Bass",
            "bar": bar,
            "beat_in_bar": beat_in_bar,
            "start_beat": bar * METER[0] + beat_in_bar,
            "degree": degree,
            "pitch": TONIC_MIDI + degree_offset(degree, SCALE_INTERVALS) - 12,
            "duration_beats": duration_beats,
            "velocity": velocity,
        })

print("Lead notes:", sum(e["part"] == "Lead" for e in events))
print("Bass notes:", sum(e["part"] == "Bass" for e in events))

## Step 4 — Write the MIDI variants

Each MIDI file has a tempo/meter conductor track plus separate Lead and Bass tracks. For the instrument comparison, only the Lead program number changes. A piano version can also be exported with only one part at a time for later editing.

In [ ]:
def vlq(value):
    value = int(value)
    if value < 0:
        raise ValueError("MIDI delta times cannot be negative")
    encoded = [value & 0x7F]
    value >>= 7
    while value:
        encoded.insert(0, (value & 0x7F) | 0x80)
        value >>= 7
    return bytes(encoded)

def midi_meta(kind, payload):
    return bytes([0xFF, kind]) + vlq(len(payload)) + payload

def make_track(track_name, channel, part_events=None, program=None, tempo=None, meter=None):
    timed = []
    if track_name:
        timed.append((0, -3, midi_meta(0x03, track_name.encode("ascii"))))
    if tempo is not None:
        micros = round(60_000_000 / tempo)
        timed.append((0, -2, midi_meta(0x51, micros.to_bytes(3, "big"))))
    if meter is not None:
        numerator, denominator = meter
        denominator_power = int(round(math.log2(denominator)))
        timed.append((0, -1, midi_meta(0x58, bytes([numerator, denominator_power, 24, 8]))))
    if program is not None:
        timed.append((0, 0, bytes([0xC0 | channel, program])))

    for ev in part_events or []:
        start_tick = round(ev["start_beat"] * TICKS_PER_BEAT)
        end_tick = round((ev["start_beat"] + ev["duration_beats"]) * TICKS_PER_BEAT)
        pitch = int(ev["pitch"])
        velocity = int(np.clip(ev["velocity"], 1, 127))
        timed.append((start_tick, 1, bytes([0x90 | channel, pitch, velocity])))
        timed.append((end_tick, 0, bytes([0x80 | channel, pitch, 0])))

    timed.sort(key=lambda item: (item[0], item[1]))
    body = bytearray()
    previous_tick = 0
    for tick, _, payload in timed:
        body.extend(vlq(tick - previous_tick))
        body.extend(payload)
        previous_tick = tick
    body.extend(b"\x00\xFF\x2F\x00")
    return b"MTrk" + len(body).to_bytes(4, "big") + body

def write_midi(path, lead_program, bass_program=BASS_PROGRAM, parts=("Lead", "Bass")):
    tracks = [make_track("Conductor", 0, tempo=BPM, meter=METER)]
    if "Lead" in parts:
        lead_events = [e for e in events if e["part"] == "Lead"]
        tracks.append(make_track("Lead", 0, lead_events, program=lead_program))
    if "Bass" in parts:
        bass_events = [e for e in events if e["part"] == "Bass"]
        tracks.append(make_track("Bass", 1, bass_events, program=bass_program))
    header = b"MThd" + (6).to_bytes(4, "big") + struct.pack(">HHH", 1, len(tracks), TICKS_PER_BEAT)
    Path(path).write_bytes(header + b"".join(tracks))
    return Path(path)

midi_files = {}
for preset_name, program in LEAD_PRESETS.items():
    path = OUT_DIR / f"C_major_{BPM}bpm_{preset_name}_bass.mid"
    midi_files[preset_name] = write_midi(path, lead_program=program)
    print(path.name, "| bytes:", path.stat().st_size)

piano_lead_mid = write_midi(
    OUT_DIR / f"C_major_{BPM}bpm_piano_lead_only.mid",
    lead_program=LEAD_PRESETS["acoustic_piano"],
    parts=("Lead",),
)
piano_bass_mid = write_midi(
    OUT_DIR / f"C_major_{BPM}bpm_acoustic_bass_only.mid",
    lead_program=LEAD_PRESETS["acoustic_piano"],
    parts=("Bass",),
)
print("Separate MIDI stems:", piano_lead_mid.name, "and", piano_bass_mid.name)

## Step 5 — Render the MIDI files with FluidSynth

FluidSynth reads the MIDI note and program events and renders them through the SoundFont. The render is deterministic for a fixed MIDI, SoundFont, sample rate, and FluidSynth settings.

We render the three full arrangements, then render the piano Lead and Acoustic Bass separately. The MIDI remains editable after rendering.

In [ ]:
def render_midi(midi_path, wav_path, disable_effects=False):
    command = ["fluidsynth", "-ni", "-F", str(wav_path), "-r", str(SAMPLE_RATE)]
    if disable_effects:
        command.extend(["-C", "0", "-R", "0"])
    command.extend([str(SOUNDFONT), str(midi_path)])
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode != 0:
        print(result.stdout[-2000:])
        print(result.stderr[-4000:])
        raise RuntimeError(f"FluidSynth render failed for {midi_path.name}")
    if not Path(wav_path).exists():
        raise FileNotFoundError(f"Expected WAV was not created: {wav_path}")
    return Path(wav_path)

wav_files = {}
for preset_name, midi_path in midi_files.items():
    out = OUT_DIR / f"{preset_name}_mix.wav"
    wav_files[preset_name] = render_midi(midi_path, out)

piano_lead_wav = render_midi(
    piano_lead_mid, OUT_DIR / "piano_lead_stem.wav"
)
piano_bass_wav = render_midi(
    piano_bass_mid, OUT_DIR / "acoustic_bass_stem.wav"
)

print("Rendered files:")
for path in [*wav_files.values(), piano_lead_wav, piano_bass_wav]:
    print("-", path.name, f"({path.stat().st_size / (1024**2):.2f} MiB)")

## Step 6 — Listen to the controlled comparison

Listen for instrument character and balance. Since the same note-event list is used in each mix, a different melody or rhythm should not appear when only the Lead preset changes.

In [ ]:
def listen_wav(path):
    sample_rate, audio = read_wav(str(path))
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    elif audio.dtype == np.int32:
        audio = audio.astype(np.float32) / 2147483648.0
    else:
        audio = audio.astype(np.float32)
        peak = float(np.max(np.abs(audio))) if audio.size else 0.0
        if peak > 1:
            audio = audio / peak
    print(path.name, "| sample rate:", sample_rate, "| shape:", audio.shape)
    display(Audio(audio, rate=sample_rate))

for preset_name in LEAD_PRESETS:
    print(f"Lead patch: {preset_name}; Bass patch: acoustic bass")
    listen_wav(wav_files[preset_name])

print("Piano lead stem:")
listen_wav(piano_lead_wav)
print("Acoustic bass stem:")
listen_wav(piano_bass_wav)

## Step 7 — Preserve the exact controls and licensing evidence

The manifest records the MIDI settings, note events, program numbers, SoundFont file hash, package versions, and output paths. The Debian copyright notice is copied into the ZIP so the source and license remain attached to the experiment results.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

package_copyright_paths = [
    Path("/usr/share/doc/fluid-soundfont-gm/copyright"),
    Path("/usr/share/doc/fluidsynth/copyright"),
]
license_copies = []
for source in package_copyright_paths:
    if source.exists():
        target = OUT_DIR / f"{source.parent.name}_copyright.txt"
        target.write_text(source.read_text(errors="replace"), encoding="utf-8")
        license_copies.append(target.name)

manifest = {
    "experiment": "fluidsynth_midi_timbre_pilot_v1",
    "arrangement": {
        "key": f"{KEY_NAME} {SCALE_NAME}",
        "tonic_midi": TONIC_MIDI,
        "scale_intervals": SCALE_INTERVALS,
        "tempo_bpm": BPM,
        "meter": list(METER),
        "bars": BARS,
        "sample_rate_hz": SAMPLE_RATE,
    },
    "same_note_events_for_all_lead_preset_variants": True,
    "lead_programs_zero_based_GM": LEAD_PRESETS,
    "bass_program_zero_based_GM": BASS_PROGRAM,
    "event_count": len(events),
    "events": events,
    "renderer": {
        "name": "FluidSynth",
        "version": subprocess.check_output(["fluidsynth", "--version"], text=True).splitlines()[0],
        "soundfont": str(SOUNDFONT),
        "soundfont_sha256": sha256_file(SOUNDFONT),
        "soundfont_license_claim": "MIT, with package copyright notice included",
        "soundfont_license_source": "https://github.com/pianobooster/fluid-soundfont",
        "fluidsynth_license": "LGPL-2.1",
        "package_copyright_files": license_copies,
    },
    "outputs": {
        **{name: path.name for name, path in wav_files.items()},
        "piano_lead_stem": piano_lead_wav.name,
        "acoustic_bass_stem": piano_bass_wav.name,
        "midi_variants": {name: path.name for name, path in midi_files.items()},
        "piano_lead_midi": piano_lead_mid.name,
        "acoustic_bass_midi": piano_bass_mid.name,
    },
}
manifest_path = OUT_DIR / "render_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

bundle = Path("/content/fluidsynth_midi_timbre_pilot.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT_DIR.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print("Bundle ready:", bundle, "| size:", f"{bundle.stat().st_size / (1024**2):.2f} MiB")
print("SoundFont SHA-256:", manifest["renderer"]["soundfont_sha256"])
from google.colab import files
files.download(str(bundle))

## How to read the result

- If the three mixes have the same notes and beat positions but distinct timbres, the MIDI-to-instrument rendering stage provides exact musical control.
- If the sound is still too synthetic or dated, that is a limitation of this particular General MIDI SoundFont, not of the MIDI note/scale/rhythm controls.
- The next quality step would be to test a higher-quality instrument library whose license and sample provenance are suitable for the product. We should not assume that a clearer license automatically means a modern or realistic sound.